In [1]:
import os
from pyspark.sql import SparkSession

# 1. AWS 자격증명 프로필 설정 (터미널에서 -e AWS_PROFILE=metacode 한 것과 동일한 효과)
os.environ["AWS_PROFILE"] = "metacode"

# 2. Spark Session 생성 및 Iceberg/Glue 환경 세팅
spark = (
    SparkSession.builder.appName("Iceberg-Jupyter")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://metacode-iceberg-0426/warehouse")
    .getOrCreate()
)

print("Spark Iceberg 환경 세팅 완료!")

Spark Iceberg 환경 세팅 완료!


In [4]:
# Silver 테이블 데이터 5줄 확인
spark.sql("SELECT * FROM glue_catalog.ad_lakehouse.processed_events LIMIT 5").show()

# 파티션(날짜)별 데이터 건수 확인
spark.sql("SELECT event_date, COUNT(*) as cnt FROM glue_catalog.ad_lakehouse.processed_events GROUP BY event_date ORDER BY event_date DESC").show()

+------------+----------+--------+--------+-----+----------+--------------------+----------------+--------------------+
|    event_id|event_date|     uid|campaign|click|conversion|conversion_delay_sec|            cost|          updated_at|
+------------+----------+--------+--------+-----+----------+--------------------+----------------+--------------------+
|evt_00000712|2026-04-01|18750238|25920690|    0|         0|                NULL|8.37021415681E-4|2026-04-30 11:11:...|
|evt_00000713|2026-04-01|11410280|15654890|    0|         0|                NULL|          1.0E-5|2026-04-30 11:11:...|
|evt_00000714|2026-04-01| 6507341|15885288|    0|         0|                NULL|4.77762898131E-4|2026-04-30 11:11:...|
|evt_00000715|2026-04-01|11477114|17288262|    0|         0|                NULL|3.97058830541E-4|2026-04-30 11:11:...|
|evt_00000716|2026-04-01|14240056|31772643|    1|         0|                NULL|3.74999992988E-5|2026-04-30 11:11:...|
+------------+----------+--------+------

In [5]:
# Silver 테이블 스냅샷 이력 조회
spark.sql("""
    SELECT committed_at, snapshot_id, operation, summary 
    FROM glue_catalog.ad_lakehouse.processed_events.snapshots
""").show(truncate=False)

+-----------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                                 |
+-----------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# 1. Silver 테이블 샘플 데이터 조회
spark.sql("SELECT * FROM glue_catalog.ad_lakehouse.processed_events LIMIT 5").show()

# 2. 파티션(날짜)별 데이터 적재 건수 확인
spark.sql("""
    SELECT event_date, COUNT(*) as cnt 
    FROM glue_catalog.ad_lakehouse.processed_events 
    GROUP BY event_date 
    ORDER BY event_date DESC
""").show()

In [6]:
# 1. Gold 테이블 샘플 데이터 확인 (전환수 기준 내림차순 5건)
spark.sql("""
    SELECT * 
    FROM glue_catalog.ad_lakehouse.campaign_summary 
    ORDER BY conversions DESC 
    LIMIT 5
""").show()

# 2. 일자별 캠페인 집계 현황 확인
spark.sql("""
    SELECT summary_date, COUNT(*) as campaign_cnt, SUM(conversions) as total_conv
    FROM glue_catalog.ad_lakehouse.campaign_summary 
    GROUP BY summary_date 
    ORDER BY summary_date DESC
""").show()

+------------+--------+-----------+------+-----------+--------------------+-----------------+-----------------+--------------------+--------------------+
|summary_date|campaign|impressions|clicks|conversions|          total_cost|              ctr|              cvr|                 cpa|          updated_at|
+------------+--------+-----------+------+-----------+--------------------+-----------------+-----------------+--------------------+--------------------+
|  2026-04-01|26321366|          2|     2|          2|    4.46995638224E-4|            100.0|            100.0|    2.23497819112E-4|2026-04-30 11:26:...|
|  2026-03-31|15398570|         14|     3|          2|0.001828208002904...|21.42857142857143|66.66666666666667|9.141040014522001E-4|2026-04-30 11:26:...|
|  2026-03-31|10341182|         34|    18|          2|0.012657176812576701|52.94117647058824|11.11111111111111|0.006328588406288351|2026-04-30 11:26:...|
|  2026-03-31|30491418|          6|     3|          2|0.008541367052385099| 

In [7]:
# Gold 테이블 스냅샷 이력 조회
spark.sql("""
    SELECT committed_at, snapshot_id, operation, summary 
    FROM glue_catalog.ad_lakehouse.campaign_summary.snapshots
""").show(truncate=False)

+-----------------------+-------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                               |
+-----------------------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------